# E25 — QUICK diagnostic (HandReach / MAE-B / seed 0)
Run All on the existing GPU session. Stop any running official collection first; do not reset the runtime. Reuses the first 4 completed 256-episode shards or generates only missing shards. Encodes directly from cached frames (no merged RGB copy), fits both original model architectures with 500 updates each, makes a small population, and evaluates 20 disjoint episodes per condition and arm. This is a **non-official development diagnostic**, not Table 17 or SOTA evidence. All arms share the exact same fitted model and test seeds. Results and reusable fits are separate from official outputs. One GPU; no new CUDA installation.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
from importlib.metadata import version

CACHE_ROOT = Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content') / 'e25_gpu_v3'
ROOT = CACHE_ROOT / 'quick_1024_v1'
ROOT.mkdir(parents=True, exist_ok=True)
UP = CACHE_ROOT / 'counterfactual-WM'
BUNDLE = ROOT / 'e25'
BUNDLE.mkdir(exist_ok=True)
PY = sys.executable
PIN = '5c84bba92b2ff19d41175070a61238fa1396d10d'
E25_PIN = 'e6c3dff96bd9a0592ba91e908c757282f3c74a4b'
ENV = dict(os.environ, CWM_OUTPUTS=str(ROOT/'outputs'), MUJOCO_GL='egl',
           PYOPENGL_PLATFORM='egl', SDL_VIDEODRIVER='dummy', WANDB_MODE='disabled',
           PYTHONUNBUFFERED='1', HF_HOME=str(CACHE_ROOT/'hf_cache'), E25_SOURCE=str(CACHE_ROOT/'outputs/data/hand'))
def shell(*args, cwd=ROOT, capture=False):
    args = list(map(str, args))
    print('>', ' '.join(args), flush=True)
    return subprocess.run(args, cwd=cwd, env=ENV, check=True, text=True,
                          stdout=subprocess.PIPE if capture else None).stdout

def py(*args, capture=False):
    return shell(PY, *args, cwd=UP, capture=capture)

shell('nvidia-smi')
shell(PY, '-c', "import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available(), 'CPU-only torch or no GPU: select GPU and use its CUDA-enabled Python environment'; print(torch.cuda.get_device_name(0)); print(torch.ones(4,device='cuda').sum().item())")
print('Free disk GiB:', round(shutil.disk_usage(ROOT).free/2**30, 1))

In [ ]:
# Download only the selected source revision and install dependencies for HandReach.
if not UP.exists():
    shell('git', 'clone', 'https://github.com/galilai-group/counterfactual-WM', UP)
shell('git', 'checkout', '--detach', PIN, cwd=UP)
assert shell('git', 'rev-parse', 'HEAD', cwd=UP, capture=True).strip() == PIN
# Reuse the installed runtime whenever its required versions are already present.
check = """
import json
from importlib.metadata import version, distribution
for name, expected in [('mujoco','3.11.0'),('gymnasium','1.2.3'),('gymnasium-robotics','1.4.2'),('timm','1.0.26'),('transformers','5.4.0')]:
    assert version(name)==expected, (name,version(name),expected)
url=json.loads(distribution('stable-pretraining').read_text('direct_url.json'))
assert url['vcs_info']['commit_id']=='ab836bf699a2be712dfcf980a5eb70d36391e876'
from cwm.model import WorldModel
from cwm import train
"""
try:
    py('-c', check)
    print('Existing dependencies reused.')
except subprocess.CalledProcessError:
    constraints = ROOT/'constraints.txt'
    constraints.write_text('\n'.join(f'{p}=={version(p)}' for p in ('torch','torchvision','numpy'))+'\n')
    py('-m','pip','install','-q','-c',constraints,
       'mujoco==3.11.0','gymnasium==1.2.3','gymnasium-robotics==1.4.2',
       'timm==1.0.26','transformers==5.4.0',
       'stable-pretraining @ git+https://github.com/galilai-group/stable-pretraining@ab836bf699a2be712dfcf980a5eb70d36391e876')
    py('-m','pip','install','-q','--no-deps','-e',UP)
py('-c', "import torch; from cwm.model import WorldModel; from cwm import train; assert torch.cuda.is_available(); print('Imports and GPU OK')")

In [ ]:
# Fetch the committed E25 sources. Cache the posterior solve once per observation.
import urllib.request
for name in ('e25.py','run_e25.py','score_e25.py','test_e25.py'):
    url = f'https://raw.githubusercontent.com/shekhawatxyz/fastai-2022/{E25_PIN}/experiments/recursive_world/e25/{name}'
    with urllib.request.urlopen(url, timeout=60) as r:
        (BUNDLE/name).write_bytes(r.read())
p = BUNDLE/'e25.py'
s = p.read_text()
def replace_once(s, old, new):
    assert s.count(old) == 1, f'Unexpected source revision: {old}'
    return s.replace(old, new, 1)
s = replace_once(s, 'self.raw_last = torch.zeros_like(self.sigma)',
                 'self.raw_last = torch.zeros_like(self.sigma)\n        self._cached = None')
s = replace_once(s, 'action = action.reshape(-1)',
                 'self._cached = None\n        action = action.reshape(-1)')
s = replace_once(s, 'p = self.sigma.new_zeros(2*self.a+self.d)\n        p[self.mask] = torch.linalg.solve(self.prec, self.rhs)\n        return p',
                 'if self._cached is None:\n            p = self.sigma.new_zeros(2*self.a+self.d)\n            p[self.mask] = torch.linalg.solve(self.prec, self.rhs)\n            self._cached = p\n        return self._cached')
p.write_text(s)
# Resume complete records rather than restarting the whole expensive evaluation.
p = BUNDLE/'run_e25.py'; s = p.read_text()
s = replace_once(s, "with args.out.open('x') as f: # no overwrite or accidental duplicate episodes",
"""expected_keys = {(i,c,a) for i in range(100*args.seed,100*args.seed+args.count)
                         for c in design['conditions'] for a in arms}
        done = set()
        if args.out.exists():
            raw = args.out.read_bytes()
            complete = raw[:raw.rfind(b'\n')+1]
            for line in complete.splitlines():
                r = json.loads(line)
                key = (r['episode'],r['condition'],r['arm'])
                if (key not in expected_keys or key in done or r['mode'] != mode
                    or r['seed_base'] != design['seed_base']
                    or any(r.get(k) != v for k,v in tag.items())
                    or r['noise'] != list(map(float,noise)) or r['alpha'] != alpha):
                    raise ValueError('Incompatible previous output: use a fresh directory')
                done.add(key)
            if complete != raw: args.out.write_bytes(complete)
        with args.out.open('a') as f:""".replace("b'\n'", "b'\\n'"))
s = replace_once(s, "r=run_episode(up,sim,m,design,d['coordinates'],sigma,episode,condition,arm,noise,alpha)",
                 "if (episode,condition,arm) in done: continue\n                        r=run_episode(up,sim,m,design,d['coordinates'],sigma,episode,condition,arm,noise,alpha)")
s = replace_once(s, "f.write(json.dumps(r)+'\\n');f.flush()", "f.write(json.dumps(r)+'\\n');f.flush()\n                print(mode, episode+1, '/', args.count, flush=True)")
compile(s,str(p),'exec'); p.write_text(s)
py(BUNDLE/'test_e25.py')

In [ ]:
# Fail early if MuJoCo EGL or the pinned encoder cannot run.
py('-c', """
from cwm import registry, encoders
from cwm.sims import make
import numpy as np, torch
sim=make(registry.simulator('hand','population'))
try:
    sim.reset(2201000000)
    frame=sim.render()
    print('Rendered', frame.shape)
    assert frame.shape == (224,224,3)
finally: sim.close()
model=encoders.load('mae_cls','cuda')
z=encoders.encode(model,np.asarray([frame]),batch_size=1)
assert torch.isfinite(z).all()
print('Pinned encoder OK',z.shape)
""")
# Capture the actual installed versions for reproducibility.
(ROOT/'environment.txt').write_text(py('-m','pip','freeze',capture=True))

In [ ]:
# Only 1,024 factual episodes; reuse already-completed shards from the official cache.
# Never change the official config or merge/delete its shards.
old_out = CACHE_ROOT/'outputs'
old_data = old_out/'data/hand'
def old_py(*args):
    subprocess.run([PY,*map(str,args)],cwd=UP,
                   env=dict(ENV,CWM_OUTPUTS=str(old_out)),check=True)
if not (old_data/'rollouts.npz').exists():
    for i in range(4):
        shard=old_data/'shards'/f'{256*i:06d}'
        if not (shard/'rollouts.npz').is_file():
            old_py('experiments/2_data/collect.py','hand','--index',i)
        else:
            print('Reuse',shard.name,flush=True)

# Use original pinned encoder, no second copy of RGB data.
py('-c', """
import os
from pathlib import Path
import numpy as np
import torch
from cwm import encoders, registry, io
src=Path(os.environ['E25_SOURCE'])
dst=registry.data('hand')
features=registry.features('hand','mae_cls')
rollouts=dst/'rollouts.npz'
if not (features.exists() and rollouts.exists()):
    if (src/'rollouts.npz').exists():
        pieces=[(src,1024)]
    else:
        pieces=[(src/'shards'/f'{i:06d}',256) for i in range(0,1024,256)]
    model=encoders.load('mae_cls','cuda') if not features.exists() else None
    zs,rows=[],[]
    for folder,n in pieces:
        with np.load(folder/'rollouts.npz') as f:
            assert len(f['actions'])>=n
            rows.append({k:f[k] if k=='coordinates' else f[k][:n] for k in f.files})
        if model is not None:
            frames=np.load(folder/'frames.npy',mmap_mode='r')[:n]
            assert frames.shape==(n,4,224,224,3),frames.shape
            z=encoders.encode(model,frames.reshape(-1,224,224,3),batch_size=32)
            zs.append(z.reshape(n,4,-1))
        print('Prepared',folder,n,flush=True)
    if not rollouts.exists():
        io.savez(rollouts,coordinates=rows[0]['coordinates'],**{
            k:np.concatenate([r[k] for r in rows]) for k in rows[0] if k!='coordinates'})
    if model is not None:
        io.save(dict(z=torch.cat(zs),encoder='mae_cls',state_sha256=model.spec['state_sha256']),features)
with np.load(rollouts) as r:
    assert len(r['actions'])==1024
assert io.load(features)['z'].shape[:2]==(1024,4)
print('Cached 1,024 episodes and features.')
""")

# The upstream default is 30k readout + 20k predictor updates. Use 500 each for this diagnostic.
py('-c', """
import numpy as np
from cwm import registry, io, readout, predictor
d=registry.dataset('hand')
z=io.load(registry.features('hand','mae_cls'))['z']
r=np.load(registry.data('hand','rollouts.npz'))
p=registry.readout('hand','mae_cls','state')
if not p.exists():
    readout.fit(p.parent,'state',z,r['actions'],r['physical'][...,d['coordinates']],r['epsilon'],
                anchor_time=d['anchor'],hidden=[1024,1024,1024,len(d['coordinates'])],
                steps=500,validation_interval=100,device='cuda')
print('Readout',io.read_json(p.parent/'results.json'),flush=True)
f=registry.OUT/'fits/hand_mae_cls_h1_s0'
if not (f/'fit.pt').exists():
    predictor.fit(f,z,r['actions'],history=1,anchor_time=d['anchor'],anchor_only=False,
                  hidden=[1024,1024],weight_decay=.3,seed=0,steps=500,
                  validation_interval=100,device='cuda')
print('Predictor',io.read_json(f/'fit.json'),flush=True)
""")


In [ ]:
# Reduced diagnostic population: 16 parents x 16 draws x 4 actions; discard RGB writes.
# The planner consumes only branches.npz, not population frames.npy.
py('-c', """
from pathlib import Path
import numpy as np
from cwm import registry
from cwm.sims import population
out=registry.data('hand','population')
if not (out/'results.json').exists():
    import shutil
    shutil.rmtree(out, ignore_errors=True)
    original=np.lib.format.open_memmap
    class FrameSink:
        def __setitem__(self,key,value): pass
        def flush(self): pass
    def open_memmap(filename,*args,**kwargs):
        if Path(filename).resolve()==(out/'frames.npy').resolve(): return FrameSink()
        return original(filename,*args,**kwargs)
    np.lib.format.open_memmap=open_memmap
    try: population.run(dict(registry.simulator('hand','population'),parents=16,fields=16),out)
    finally: np.lib.format.open_memmap=original
assert np.load(out/'branches.npz')['truth'].shape[:3]==(16,16,4)
print('Population complete')
""")

In [ ]:
# Compare E25's two unchanged arms with the upstream Planner on disjoint smoke seeds.
# Only implementation parity is checked here, not E25's research hypothesis.
py('-c', f"""
import sys, importlib.util, numpy as np
from pathlib import Path
sys.path.insert(0,{str(BUNDLE)!r})
from cwm import registry, io
from cwm.model import WorldModel
from cwm.sims.worker import Remote
from run_e25 import run_episode
spec=importlib.util.spec_from_file_location('planning', 'experiments/4_evaluate/planning.py')
up=importlib.util.module_from_spec(spec); spec.loader.exec_module(up)
d=io.read_json(registry.CONFIGS/'planning/replan.json'); d['seed_base']+=2000000
fit=registry.OUT/'fits/hand_mae_cls_h1_s0/fit.pt'
m=WorldModel('mae_cls',fit,registry.readout('hand','mae_cls','state'),'cuda')
sigma=up.sigma_of('hand'); idx=registry.dataset('hand')['coordinates']
sim=Remote(registry.simulator('hand','population'),up.sim_command('hand'))
try:
    planner=up.Planner(sim,m,d,idx,sigma)
    for condition in d['conditions']:
        for arm in d['arms']:
            goal=sim('fix',np.random.default_rng(np.random.SeedSequence([d['seed_base'],0,14])).uniform(-planner.limit,planner.limit,m.a))
            ref=planner.replan(0,condition,arm,goal)
            row=run_episode(up,sim,m,d,idx,sigma,0,condition,arm,np.ones_like(sigma),0.)
            np.testing.assert_allclose(row['distances'],ref['distances'],atol=1e-5,rtol=0)
    print('Six smoke baseline parity checks passed')
finally: sim.close()
""")

In [ ]:
# Disjoint development seeds, not the official test seeds. 60 calibration + 360 test records.
p=BUNDLE/'run_e25.py'
s=p.read_text()
old="original_base=design['seed_base'];"
new="design['seed_base']+=3000000; original_base=design['seed_base'];"
assert s.count(old)==1
quick=BUNDLE/'run_quick.py'
quick.write_text(s.replace(old,new))
compile(quick.read_text(),str(quick),'exec')
cal=ROOT/'calibration/hand_quick.jsonl'
test=ROOT/'test/hand_quick.jsonl'
base=[quick,'--upstream',UP,'--dataset','hand','--encoder','mae_cls','--seed',0,'--device','cuda','--count',20]
py(*base,'--out',cal)
py(*base,'--calibration',cal,'--out',test)

# No official gate: check complete paired cohort, show success and mean distance for every arm.
import itertools, math
arms=('trained_predictor','counterfactual','scalar','bias','actuator','joint')
conditions=('persistent','fresh','none')
rows={}
for line in test.read_text().splitlines():
    r=json.loads(line)
    k=(r['episode'],r['condition'],r['arm'])
    assert k not in rows
    assert r['mode']=='test' and r['seed_base']==3112000000
    assert len(r['distances'])==3 and all(math.isfinite(x) for x in r['distances'])
    assert r['final_distance']==r['distances'][-1]
    assert r['success']==(r['final_distance']<.2)
    rows[k]=r
assert set(rows)==set(itertools.product(range(20),conditions,arms))
report={'scope':'development_only','training_episodes':1024,'updates_per_fit':500,'population_parents':16,
        'population_draws':16,'n_test':len(rows),'metrics':{}}
for c in conditions:
    report['metrics'][c]={}
    for a in arms:
        rs=[rows[i,c,a] for i in range(20)]
        report['metrics'][c][a]={'success_pct':100*sum(r['success'] for r in rs)/20,
                                 'mean_distance':sum(r['final_distance'] for r in rs)/20}
(ROOT/'score.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
print('Development evidence only; do not claim published benchmark parity or SOTA.')


In [ ]:
# Export the evidence (also works after partial execution).
import zipfile
archive=ROOT/'e25_quick_evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in (BUNDLE,ROOT/'calibration',ROOT/'test',ROOT/'outputs/planning/replan'):
        if folder.exists():
            for p in folder.rglob('*'):
                if p.is_file() and p.suffix in ('.py','.jsonl','.json'):
                    z.write(p,p.relative_to(ROOT))
    for rel in ('data/hand/rollouts.npz','data/hand/features/mae_cls.pt','data/hand/population/branches.npz','data/hand/population/results.json','data/hand/population/resolved.json','fits/hand_mae_cls_h1_s0/fit.pt','fits/hand_mae_cls_h1_s0/fit.json','readouts/hand/mae_cls/state/readout.pt','readouts/hand/mae_cls/state/results.json'):
        p=ROOT/'outputs'/rel
        if p.exists(): z.write(p,p.relative_to(ROOT))
    for p in (ROOT/'environment.txt',ROOT/'score.json'):
        if p.exists(): z.write(p,p.name)
from IPython.display import FileLink, display
print(archive)
display(FileLink(str(archive)))